# attention.ipynb

目標：HTML → Markdown (text-only)，丟 ollama **修補**壞掉的公式 region，輸出 LaTeX。

流程：
1. **Triage（regex）** — 預掃 markdown 找 candidate：
   - `formula_not_decoded`：region 是 `[FORMULA: not decoded]` placeholder
   - `formula_as_text`：text region 內出現被攤平的公式片段（`m t + 1`、`/Theta1`、`R n x`...）
2. **Fix（LLM, per-candidate）** — 每個 candidate 帶 minimal context window 單獨送 LLM，回傳 LaTeX。Local 模型在小 context 下準確度比一次塞整頁高，而且 chunk 之間獨立可並行。

純文字模型，無圖時靠 surrounding context 做 best-effort，confidence 標明可信度。之後若要 escalate，high/medium 直接收，low 再丟 vision agent 覆核。

In [15]:
HTML_PATH = "../outputs-step2/pages/page_04/rendered.html"
MODEL_ID = "qwen3.5:9b"
SOURCE_PAGE = 4  # 註：之後可從 data-source-page 自動抓

## 1. HTML → Markdown

策略：用 BeautifulSoup 找所有 `data-region-id` 的 div，每個 region 轉成一個 markdown block，前面掛 `<!-- region-XXXX | label -->` comment 當 anchor，這樣 LLM 抓錯的時候可以直接引 region_id。

In [16]:
from pathlib import Path
from bs4 import BeautifulSoup, Tag

FORMULA_PLACEHOLDER = "[FORMULA: not decoded]"


def _region_to_md(region: Tag) -> str:
    """把單一 region div 轉成 markdown block。
    label 決定怎麼渲染：formula → placeholder 或內容；section_header → ##；其餘 → 段落純文字。
    """
    label = region.get("data-label", "text")
    region_id = region.get("data-region-id", "unknown")
    header = f"<!-- {region_id} | {label} -->"

    if label == "formula":
        not_decoded = region.find(class_="formula-not-decoded")
        if not_decoded is not None:
            body = FORMULA_PLACEHOLDER
        else:
            # 已 decoded 的公式 → 直接吐文字內容（之後 LLM 還是會看到）
            body = region.get_text(" ", strip=True)
    elif label == "section_header":
        h = region.find(["h1", "h2", "h3", "h4", "h5", "h6"])
        level = int(h.name[1]) if h else 2
        body = "#" * level + " " + (h.get_text(" ", strip=True) if h else "")
    elif label in ("page_header", "page_footer"):
        body = "> " + region.get_text(" ", strip=True)
    elif label == "footnote":
        body = "^ " + region.get_text(" ", strip=True)
    else:  # text / 其他
        body = region.get_text(" ", strip=True)

    return f"{header}\n{body}"


def html_to_markdown(html_path: str | Path) -> str:
    html = Path(html_path).read_text(encoding="utf-8")
    soup = BeautifulSoup(html, "html.parser")
    page = soup.find("div", class_="page")
    if page is None:
        page = soup.body
    regions = page.find_all("div", attrs={"data-region-id": True}, recursive=True)
    # 只取 top-level region（不要把 formula 內的 inner div 也抓出來）
    regions = [r for r in regions if r.find_parent("div", attrs={"data-region-id": True}) is None]
    return "\n\n".join(_region_to_md(r) for r in regions)


md = html_to_markdown(HTML_PATH)
print(f"--- markdown ({len(md)} chars) ---\n")
print(md)

--- markdown (4456 chars) ---

<!-- region-0051 | page_header -->
> L. Maliar, S. Maliar and P. Winant

<!-- region-0052 | page_header -->
> Journal of Monetary Economics 122 (2021) 76-101

<!-- region-0053 | section_header -->
## 2.1. A class of dynamic economic models

<!-- region-0054 | text -->
We consider a class of dynamic Markov economic models with time-invariant decision functions - the main framework in modern economic dynamics. An agent (consumer, firm, government, central bank, etc.) solves a canonical intertemporal optimization problem. 4

<!-- region-0055 | text -->
Definition 2.1 (Optimization problem) . An exogenous state m t + 1 ∈ R n m follows a Markov process driven by an i.i.d. innovation process ϵ t ∈ R m with a transition function M,

<!-- region-0056 | formula -->
[FORMULA: not decoded]

<!-- region-0057 | text -->
An endogenous state s t + 1 is driven by the exogenous state m t and controlled by a choice x t ∈ R n x according to a transition function S,

<!-- re

## 2. Pydantic schema

每個 fix 必須掛回 markdown 裡那個 `region-XXXX`，附上重建後的 LaTeX、原始 source、confidence。

In [17]:
from typing import Literal
from pydantic import BaseModel, Field


class FormulaFix(BaseModel):
    region_id: str = Field(description="必須是 markdown 裡出現過的 region-XXXX")
    type: Literal[
        "formula_not_decoded",
        "formula_as_text",
        "formula_garbled",
    ]
    latex: str = Field(description="重建後的 LaTeX；純 body，不包 $$、\\[ \\]、\\begin{equation}")
    confidence: Literal["high", "medium", "low"]
    source: str = Field(description="原始片段或 placeholder（從 markdown 直接複製，<=120 字）")
    reason: str = Field(description="根據哪段上下文 / 哪個 region / 哪些 token 推導出 LaTeX")


class FixBatch(BaseModel):
    """每次 LLM call 回傳一批 fixes（formula_not_decoded 通常 1 個；formula_as_text 0~N 個）。"""
    fixes: list[FormulaFix]


class FormulaFixReport(BaseModel):
    """所有 candidate 跑完後 aggregated 的結果。"""
    source_page: int
    fixes: list[FormulaFix]

## 3. Triage (regex 預掃)

用 regex 把 markdown 切成 region，再找兩類 candidate：
- `formula_not_decoded` → 固定字串 `[FORMULA: not decoded]`，定位精準
- `formula_as_text` → 啟發式 pattern（變數加空白、`/Theta1`、希臘字母+下標散開），可能有偽陽性，LLM 端會二次過濾

每個 candidate 帶一個 `window`：
- `formula_not_decoded`：[前一個 text region, 目標 formula, 後一個 text region]
- `formula_as_text`：只放目標 region 本身（資訊都在裡面）

In [18]:
import re

REGION_HEADER = re.compile(r"<!-- (region-\d+) \| (\w+) -->")

# 啟發式：text region 內被攤平的公式片段
SPACED_MATH = re.compile(
    r"\b[a-zA-Z]\s+[tT]\s*[+\-]\s*\d"                                # x t + 1
    r"|\bR\s+n\s*[_a-zA-Z]"                                            # R n x / R n_x
    r"|/(?:Theta|Alpha|Beta|Gamma|Delta|Epsilon|Phi|Psi|Omega)\d?"     # /Theta1
    r"|[ϵβθϕψαγδ]\s+[tT0-9]\b"                                         # ϵ t / θ 0
    r"|\(\s*[a-zA-Zϵβθϕψ]\s+[tT]\s*,"                                  # ( m t , ...
    r"|\bE\s+\d\s*\["                                                  # E 0 [
)


def parse_regions(md: str) -> list[tuple[str, str, str]]:
    """切成 [(region_id, label, body), ...]"""
    parts = REGION_HEADER.split(md)
    out = []
    for i in range(1, len(parts), 3):
        rid, label, body = parts[i], parts[i + 1], parts[i + 2].strip()
        out.append((rid, label, body))
    return out


def triage(md: str) -> list[dict]:
    """掃 markdown 找 candidate，每個帶 minimal context window。"""
    regions = parse_regions(md)
    candidates: list[dict] = []
    for i, (rid, label, body) in enumerate(regions):
        if label == "formula" and FORMULA_PLACEHOLDER in body:
            prev = next(((r, l, b) for r, l, b in reversed(regions[:i]) if l in ("text", "section_header")), None)
            nxt = next(((r, l, b) for r, l, b in regions[i + 1:] if l in ("text", "section_header")), None)
            window = [x for x in [prev, (rid, label, body), nxt] if x is not None]
            candidates.append({"region_id": rid, "type": "formula_not_decoded", "window": window})
        elif label == "text" and SPACED_MATH.search(body):
            candidates.append({"region_id": rid, "type": "formula_as_text", "window": [(rid, label, body)]})
    return candidates


_regions = parse_regions(md)
candidates = triage(md)
print(f"parsed {len(_regions)} regions → {len(candidates)} candidate(s)\n")
for c in candidates:
    win_ids = [w[0] for w in c["window"]]
    print(f"  {c['region_id']:14}  {c['type']:22}  window={win_ids}")

parsed 26 regions → 14 candidate(s)

  region-0055     formula_as_text         window=['region-0055']
  region-0056     formula_not_decoded     window=['region-0055', 'region-0056', 'region-0057']
  region-0057     formula_as_text         window=['region-0057']
  region-0058     formula_not_decoded     window=['region-0057', 'region-0058', 'region-0059']
  region-0060     formula_not_decoded     window=['region-0059', 'region-0060', 'region-0061']
  region-0061     formula_as_text         window=['region-0061']
  region-0062     formula_not_decoded     window=['region-0061', 'region-0062', 'region-0063']
  region-0063     formula_as_text         window=['region-0063']
  region-0064     formula_as_text         window=['region-0064']
  region-0065     formula_as_text         window=['region-0065']
  region-0066     formula_as_text         window=['region-0066']
  region-0069     formula_as_text         window=['region-0069']
  region-0070     formula_not_decoded     window=['region-0069'

## 4. Prompts

兩個專用 prompt：每個 candidate 只送對應類型的 prompt + minimal window。LLM 不再做 detection，只負責 LaTeX 重建。

In [19]:
NOT_DECODED_PROMPT = """你是公式重建 agent。

輸入：一個 `[FORMULA: not decoded]` placeholder region，加上前後最多各一個 text/section_header region 當 context。

任務：從前後文推斷該方程式內容，輸出 LaTeX。

學術論文慣例 — text region 通常用文字描述後面 formula 的角色：
- "An exogenous state m_{t+1} follows a Markov process with transition function M" → `m_{t+1} = M(m_t, \\epsilon_{t+1})`
- "agent maximizes discounted lifetime reward" → `\\max \\mathbb{E}_0 \\sum_{t=0}^{\\infty} \\beta^t r(m_t, s_t, x_t)`
- "value function V is the maximum expected lifetime reward" → `V(m_0, s_0) = \\max \\mathbb{E}_0 \\sum_{t=0}^{T} \\beta^t r(m_t, s_t, x_t)`

**LaTeX 規格**：
- 純 body，不包 `$`、`$$`、`\\[ \\]`、`\\begin{equation}`
- 下標 `_{...}`、上標 `^{...}`，多字元一律加大括號
- 巨集：`\\mathbb{R}^{n_x}`、`\\phi(\\cdot; \\theta)`、`\\sum_{t=0}^{T}`、`\\mathbb{E}_0[\\cdot]`、`\\max`、`\\in`
- 希臘字母：`\\beta`、`\\epsilon`、`\\theta`、`\\phi`、`\\Theta`，不要 unicode
- 多行用 `\\begin{aligned} ... \\end{aligned}`，`\\\\` 換行

**Confidence**：
- `high`：上下文明確指出方程式
- `medium`：能推主體，個別下標需猜
- `low`：上下文不足，best guess

不確定也要輸出，**不要拒答**、不要回空字串。

只輸出 JSON，無 fence：
{
  "fixes": [
    {
      "region_id": "<目標 region_id>",
      "type": "formula_not_decoded",
      "latex": "<LaTeX>",
      "confidence": "high | medium | low",
      "source": "[FORMULA: not decoded]",
      "reason": "<引用哪個 region 推出的>"
    }
  ]
}
"""

AS_TEXT_PROMPT = """你是公式重建 agent。

輸入：一個 text region，內容含有一個或多個被攤平成純文字的方程式片段。特徵：
- 變數加空白：`m t + 1`、`R n x`、`x t`、`( m t , s t )`
- LaTeX 殘骸：`/Theta1`、`/beta`、`/alpha`
- 希臘字母 + 上下標散開：`ϵ t ∈ R m`

任務：把每個方程式片段重組為 LaTeX，每個片段一個 fix。

**重要**：純粹引用變數名（"the function ϕ"、"discount factor β"、"vector θ"）**不算公式**，跳過。只挑像被攤平方程式的片段。

**LaTeX 規格**：
- 純 body，不包 `$`、`$$`、`\\[ \\]`、`\\begin{equation}`
- 下標 `_{...}`、上標 `^{...}`，多字元一律加大括號
- 巨集：`\\mathbb{R}^{n_x}`、`\\phi(\\cdot; \\theta)`、`\\sum_{t=0}^{T}`、`\\mathbb{E}_0[\\cdot]`、`\\in`
- 希臘字母：`\\beta`、`\\epsilon`、`\\theta`、`\\phi`、`\\Theta`，不要 unicode

**Confidence**：source 資訊完整 → `high`；個別 token 需猜 → `medium`。

只輸出 JSON，無 fence：
{
  "fixes": [
    {
      "region_id": "<region_id>",
      "type": "formula_as_text",
      "latex": "<LaTeX>",
      "confidence": "high | medium | low",
      "source": "<從原文複製的片段，<=120 字>",
      "reason": "<為什麼判定為公式>"
    }
  ]
}

若該 region 內沒有真正的方程式片段（只是變數引用） → `{"fixes": []}`。
"""

USER_TEMPLATE = """target region_id: {rid}

context window:

```markdown
{window}
```

請輸出 JSON。
"""

## 5. 跑 ollama (per-candidate)

每個 candidate 單獨呼叫 LLM；目前是序列跑。要加速可改 `asyncio` + `ollama.AsyncClient` 並行。

In [20]:
import re
import ollama

PROMPTS = {
    "formula_not_decoded": NOT_DECODED_PROMPT,
    "formula_as_text": AS_TEXT_PROMPT,
}


def _render_window(window: list[tuple[str, str, str]]) -> str:
    return "\n\n".join(f"<!-- {rid} | {label} -->\n{body}" for rid, label, body in window)


def _parse_batch(raw: str) -> FixBatch | None:
    try:
        return FixBatch.model_validate_json(raw)
    except Exception:
        pass
    m = re.search(r"\{.*\}", raw, re.DOTALL)
    if not m:
        return None
    try:
        return FixBatch.model_validate_json(m.group(0))
    except Exception:
        return None


def fix_candidate(candidate: dict, model_id: str = MODEL_ID) -> tuple[FixBatch | None, str]:
    system = PROMPTS[candidate["type"]]
    user = USER_TEMPLATE.format(
        rid=candidate["region_id"],
        window=_render_window(candidate["window"]),
    )
    client = ollama.Client(timeout=300)
    response = client.chat(
        model=model_id,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        think=False,
        options={"num_predict": 2048},
    )
    raw = response["message"]["content"]
    return _parse_batch(raw), raw


def fix_page(md: str, source_page: int, model_id: str = MODEL_ID) -> tuple[FormulaFixReport, list[tuple[dict, str]]]:
    """跑完所有 candidate，回傳 aggregated report + per-candidate raw output（debug 用）。"""
    cands = triage(md)
    all_fixes: list[FormulaFix] = []
    raws: list[tuple[dict, str]] = []
    for i, c in enumerate(cands, 1):
        print(f"[{i:>2}/{len(cands)}] {c['region_id']:14} {c['type']:22}", end=" → ")
        batch, raw = fix_candidate(c, model_id)
        raws.append((c, raw))
        if batch is None:
            print("PARSE FAIL")
            continue
        print(f"{len(batch.fixes)} fix(es)")
        all_fixes.extend(batch.fixes)
    return FormulaFixReport(source_page=source_page, fixes=all_fixes), raws


report, raws = fix_page(md, SOURCE_PAGE)
print(f"\ntotal: {len(report.fixes)} fix(es)")

[ 1/14] region-0055    formula_as_text        → 2 fix(es)
[ 2/14] region-0056    formula_not_decoded    → 1 fix(es)
[ 3/14] region-0057    formula_as_text        → 1 fix(es)
[ 4/14] region-0058    formula_not_decoded    → 1 fix(es)
[ 5/14] region-0060    formula_not_decoded    → 1 fix(es)
[ 6/14] region-0061    formula_as_text        → 0 fix(es)
[ 7/14] region-0062    formula_not_decoded    → 1 fix(es)
[ 8/14] region-0063    formula_as_text        → 4 fix(es)
[ 9/14] region-0064    formula_as_text        → 1 fix(es)
[10/14] region-0065    formula_as_text        → 4 fix(es)
[11/14] region-0066    formula_as_text        → 5 fix(es)
[12/14] region-0069    formula_as_text        → 2 fix(es)
[13/14] region-0070    formula_not_decoded    → 1 fix(es)
[14/14] region-0073    formula_not_decoded    → 1 fix(es)

total: 25 fix(es)


## 6. 結果

In [21]:
print(f"page {report.source_page} → {len(report.fixes)} formula fix(es)\n")
for i, fix in enumerate(report.fixes, 1):
    print(f"[{i}] {fix.region_id}  type={fix.type}  confidence={fix.confidence}")
    print(f"    source: {fix.source}")
    print(f"    latex : {fix.latex}")
    print(f"    reason: {fix.reason}")
    print()

page 4 → 25 formula fix(es)

[1] region-0055  type=formula_as_text  confidence=high
    source: m t + 1 ∈ R n m
    latex : m_{t+1} \in \mathbb{R}^{n_m}
    reason: Contains variables with subscripts and superscripts, set membership symbol, and vector space notation, indicating a mathematical equation.

[2] region-0055  type=formula_as_text  confidence=high
    source: ϵ t ∈ R m
    latex : \epsilon_{t} \in \mathbb{R}^{m}
    reason: Contains a Greek letter with subscript, set membership symbol, and vector space notation, indicating a mathematical equation.

[3] region-0056  type=formula_not_decoded  confidence=high
    source: region-0055
    latex : m_{t+1} = M(m_t, \epsilon_{t+1})
    reason: region-0055 describes 'm_{t+1}' as an exogenous state following a Markov process driven by innovation 'epsilon_t' with transition function 'M', implying the state transition equation.

[4] region-0057  type=formula_as_text  confidence=medium
    source: An endogenous state s t + 1 is driven by 

## 7. Sanity check

確認 LLM 抓到的 region_id 真的出現過、placeholder 有沒有漏修。

In [22]:
import re

all_regions = re.findall(r"<!-- (region-\d+) \| (\w+) -->", md)
placeholder_regions = [
    rid for rid, label in all_regions
    if label == "formula" and f"<!-- {rid} | formula -->\n{FORMULA_PLACEHOLDER}" in md
]
print(f"total regions in md      : {len(all_regions)}")
print(f"not-decoded formula regs : {len(placeholder_regions)}  → {placeholder_regions}")
if report is not None:
    fixed = {f.region_id for f in report.fixes}
    missed = set(placeholder_regions) - fixed
    extra  = fixed - {rid for rid, _ in all_regions}
    print(f"fixed by LLM             : {len(fixed)}  → {sorted(fixed)}")
    print(f"missed placeholders      : {len(missed)}   → {sorted(missed)}")
    print(f"hallucinated region_ids  : {len(extra)}    → {sorted(extra)}")

total regions in md      : 26
not-decoded formula regs : 6  → ['region-0056', 'region-0058', 'region-0060', 'region-0062', 'region-0070', 'region-0073']
fixed by LLM             : 13  → ['region-0055', 'region-0056', 'region-0057', 'region-0058', 'region-0060', 'region-0062', 'region-0063', 'region-0064', 'region-0065', 'region-0066', 'region-0069', 'region-0070', 'region-0073']
missed placeholders      : 0   → []
hallucinated region_ids  : 0    → []
